In [1]:
import librosa
import soundfile as sf

def load_audio(file_path):
    try:
       y, sr = librosa.load(file_path, sr=22050)
 # Try librosa
    except Exception:
        # Fallback if librosa fails
        y, sr = sf.read(file_path, always_2d=False)
    return y, sr


In [2]:
import numpy as np

def extract_mfcc(y, sr, n_mfcc=13):
    
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    return mfcc


In [3]:
def extract_deltas(mfcc):
    delta = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)
    return delta, delta2


In [4]:
def summarize_features(mfcc, delta, delta2):
    
    features = np.hstack([
        np.mean(mfcc, axis=1),
        np.std(mfcc, axis=1),
        np.mean(delta, axis=1),
        np.std(delta, axis=1),
        np.mean(delta2, axis=1),
        np.std(delta2, axis=1)
    ])
    return features


In [5]:
def extract_features_from_file(file_path):

    y, sr = load_audio(file_path)
    mfcc = extract_mfcc(y, sr)
    delta, delta2 = extract_deltas(mfcc)
    features = summarize_features(mfcc, delta, delta2)
    return features


In [6]:
import pandas as pd
import os

def build_dataset(base_dir, n_mfcc=13):
    classes = sorted([d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))])
    class2idx = {c:i for i,c in enumerate(classes)}
    rows = []
    bad = []

    for label in classes:
        class_dir = os.path.join(base_dir, label)
        for fname in os.listdir(class_dir):
            if not fname.lower().endswith('.wav'):
                continue
            path = os.path.join(class_dir, fname)
            try:
                features = extract_features_from_file(path)  # numeric 1D np.array
                rows.append(np.concatenate([features, [class2idx[label]]]))  # numeric label
            except Exception as e:
                bad.append((path, str(e)))

    # build column names dynamically from n_mfcc
    cols = []
    for prefix in ("mfcc_mean", "mfcc_std", "delta_mean", "delta_std", "delta2_mean", "delta2_std"):
        cols += [f"{prefix}_{i+1}" for i in range(n_mfcc)]
    cols += ["label"]

    df = pd.DataFrame(rows, columns=cols)
    return df, class2idx, bad


In [7]:
def build_test_dataset(base_dir, n_mfcc=13):
    import os, numpy as np, pandas as pd

    rows, bad = [], []
    for fname in os.listdir(base_dir):
        if not fname.lower().endswith('.wav'):
            continue
        path = os.path.join(base_dir, fname)
        try:
            features = extract_features_from_file(path)
            rows.append(features)
        except Exception as e:
            bad.append((path, str(e)))

    # build column names dynamically
    cols = []
    for prefix in ("mfcc_mean", "mfcc_std", "delta_mean", "delta_std", "delta2_mean", "delta2_std"):
        cols += [f"{prefix}_{i+1}" for i in range(n_mfcc)]

    df = pd.DataFrame(rows, columns=cols)
    return df, bad


In [8]:
TEST_DIR = r"C:\Users\sarth\Downloads\the-frequency-quest\test\test"

test_df, bad_test = build_test_dataset(TEST_DIR)
print(f"✅ Test set created successfully: {test_df.shape}")
print(f"⚠️ Skipped test files: {len(bad_test)}")
test_df.head()


✅ Test set created successfully: (739, 78)
⚠️ Skipped test files: 1


,mfcc_mean_1,mfcc_mean_2,mfcc_mean_3,mfcc_mean_4,mfcc_mean_5,mfcc_mean_6,mfcc_mean_7,mfcc_mean_8,mfcc_mean_9,mfcc_mean_10,...,delta2_std_4,delta2_std_5,delta2_std_6,delta2_std_7,delta2_std_8,delta2_std_9,delta2_std_10,delta2_std_11,delta2_std_12,delta2_std_13
0,-298.826080,103.677689,-94.644806,-13.064194,10.258672,-25.485121,-16.722870,-4.510607,-4.012282,13.754400,...,1.199600,1.486084,1.479148,0.957301,0.907906,1.035746,1.268204,1.179535,1.237090,0.878959
1,-366.390045,126.183571,31.372793,-23.601944,-12.954138,0.484963,-26.859764,-19.409349,-21.998045,-10.158172,...,4.913876,2.120577,1.158188,2.566597,1.185108,1.578759,1.064972,1.505374,1.222998,0.944294
2,-224.463379,145.606949,-1.675683,10.994343,10.010615,12.727025,-5.944498,6.288854,1.849195,-5.480206,...,0.960529,0.781848,0.623063,0.871562,0.913953,0.641937,0.783152,0.635151,0.679200,0.656191
3,-234.901459,149.483292,-1.801635,10.838568,8.305273,11.176650,-9.709814,0.757325,5.710452,-0.992717,...,0.968399,0.938530,1.173306,0.694751,0.737384,0.647341,0.601497,0.713258,0.596743,0.614641
4,-253.166641,98.504845,-41.171967,-14.738835,-8.163910,-6.949661,2.507291,-1.285358,8.260894,4.664522,...,3.761954,2.157826,1.069387,1.019672,0.748150,0.984783,1.129663,0.971911,1.213187,1.222419


In [9]:
train_dir = r"C:\Users\sarth\Downloads\the-frequency-quest\train\train"

train_df, class2idx, bad_train = build_dataset(train_dir)

print("✅ Train:", train_df.shape)
print("⚠️ Skipped train files:", len(bad_train))



✅ Train: (3448, 79)
⚠️ Skipped train files: 2


In [10]:
from sklearn.model_selection import train_test_split

# Separate features and labels
X = train_df.drop(columns=["label"])
y = train_df["label"]

# Split train data into train/validation for internal testing
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)




Train shape: (2758, 78)
Validation shape: (690, 78)


In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators': [500, 1000, 2000],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None],
    'bootstrap': [True, False]
}

rf = RandomForestClassifier(random_state=42)
search = RandomizedSearchCV(
    rf, param_distributions=param_dist,
    n_iter=30, cv=5, verbose=2, n_jobs=-1
)
search.fit(X_train, y_train)

print("Best Parameters:", search.best_params_)
print("Best Accuracy:", search.best_score_)


Fitting 5 folds for each of 30 candidates, totalling 150 fits


KeyboardInterrupt: 

In [1]:

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=2000,      # number of trees
    min_samples_split=2,    # min samples to split an internal node
    min_samples_leaf=1,     # min samples at a leaf node
    max_features='sqrt',    # features considered per split
    bootstrap=False,        # sampling without replacement
    max_depth=None,         # let trees grow fully
    n_jobs=-1,              # use all CPU cores
    random_state=42         # reproducibility
)


rf.fit(X_train, y_train)


NameError: name 'X_train' is not defined

In [ ]:
rf.predict(test_df)

array([0., 0., 4., 4., 0., 4., 2., 2., 2., 2., 3., 1., 1., 1., 2., 2., 2.,
       2., 2., 2., 1., 4., 1., 0., 0., 3., 1., 0., 2., 2., 2., 2., 2., 2.,
       2., 3., 2., 2., 3., 3., 3., 3., 1., 1., 4., 4., 0., 0., 0., 4., 4.,
       2., 2., 2., 2., 1., 1., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2.,
       4., 0., 4., 4., 0., 4., 4., 4., 4., 0., 0., 0., 0., 1., 3., 3., 3.,
       0., 4., 1., 3., 3., 3., 3., 2., 2., 2., 0., 0., 2., 0., 1., 4., 4.,
       3., 3., 3., 3., 3., 3., 4., 0., 4., 1., 1., 0., 4., 2., 0., 0., 4.,
       1., 1., 1., 0., 2., 4., 3., 3., 3., 3., 1., 4., 4., 4., 0., 1., 1.,
       1., 4., 4., 4., 1., 4., 1., 1., 1., 4., 2., 2., 2., 2., 2., 2., 2.,
       2., 0., 2., 2., 2., 2., 2., 2., 1., 1., 1., 1., 1., 1., 2., 2., 2.,
       2., 2., 4., 4., 0., 4., 0., 2., 2., 2., 2., 2., 0., 4., 4., 4., 4.,
       0., 4., 4., 4., 4., 4., 4., 4., 0., 2., 0., 0., 0., 2., 2., 0., 2.,
       2., 1., 1., 1., 2., 2., 4., 2., 2., 3., 3., 3., 3., 3., 4., 0., 3.,
       3., 3., 3., 3., 3.

In [ ]:
from joblib import dump

# Save model to file
dump(rf, "final_random_forest_model.joblib")

print("✅ Model saved successfully!")


✅ Model saved successfully!


In [ ]:
from joblib import load

# Load model from file
rf = load("final_random_forest_model.joblib")

print("✅ Model loaded successfully!")


✅ Model loaded successfully!


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = rf.predict(X_val)
print("Validation Accuracy %:", 100*accuracy_score(y_val, y_pred))
print("\nClassification Report:\n", classification_report(y_val, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_val, y_pred))


Validation Accuracy %: 90.8695652173913

Classification Report:
               precision    recall  f1-score   support

         0.0       0.91      0.83      0.87       140
         1.0       0.94      0.96      0.95       140
         2.0       0.91      0.99      0.95       140
         3.0       0.90      0.93      0.92       130
         4.0       0.88      0.84      0.86       140

    accuracy                           0.91       690
   macro avg       0.91      0.91      0.91       690
weighted avg       0.91      0.91      0.91       690


Confusion Matrix:
 [[116   5   3   3  13]
 [  2 134   1   2   1]
 [  0   1 139   0   0]
 [  2   0   5 121   2]
 [  8   2   5   8 117]]


In [ ]:
y_pred = rf.predict(X_val)

from sklearn.metrics import accuracy_score, classification_report

print("✅ Test Accuracy:", accuracy_score(y_val, y_pred))
print("\nClassification Report:\n", classification_report(y_val, y_pred))


✅ Test Accuracy: 0.908695652173913

Classification Report:
               precision    recall  f1-score   support

         0.0       0.91      0.83      0.87       140
         1.0       0.94      0.96      0.95       140
         2.0       0.91      0.99      0.95       140
         3.0       0.90      0.93      0.92       130
         4.0       0.88      0.84      0.86       140

    accuracy                           0.91       690
   macro avg       0.91      0.91      0.91       690
weighted avg       0.91      0.91      0.91       690

